# Data Pipeline: Transcribe, Translate, Synthesize, Align

**GPU Budget:** ~6-8 hours on Kaggle T4 (per direction)

This notebook runs the complete data preparation pipeline for creating
synthetic parallel speech pairs:

```
Source Audio (CV)  -->  Whisper transcribe  -->  MADLAD translate
    -->  MMS-TTS synthesize target speech  -->  Contextual align
    -->  Silence insertion  -->  Save to Kaggle working dir
```

### Prerequisites
- Common Voice Swahili uploaded as Kaggle Dataset (`cv-swahili`)
- Common Voice English uploaded as Kaggle Dataset (`cv-english`) — optional
- Repo uploaded as Kaggle Dataset (`hibiki-sw-code`)
- See `00a_stage0_vits.ipynb` for one-time setup instructions

### Run Order
Run this notebook **twice** -- once per direction:
1. First run: **Sw->En** (source=sw, target=en) -- uses pretrained `facebook/mms-tts-eng`
2. Second run: **En->Sw** (source=en, target=sw) -- uses pretrained `facebook/mms-tts-swh` (no fine-tuning needed)

**After each run:** save `/kaggle/working/hibiki-sw` output as a new Kaggle Dataset version before the session ends.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy faster-whisper sentencepiece

In [ ]:
import os
import sys
import torch

# Repo uploaded as a Kaggle Dataset (hibiki-sw-code)
REPO_DIR = '/kaggle/input/hibiki-sw-code/hibiki-sw'
sys.path.insert(0, REPO_DIR)

# Output directory — Kaggle working dir (save as Kaggle Dataset before session ends)
BASE_DIR = '/kaggle/working/hibiki-sw'
os.makedirs(BASE_DIR, exist_ok=True)

# === CONFIGURE THIS SECTION ===

# Which direction are we processing?
# First run:  SOURCE_LANG='sw', TARGET_LANG='en'
# Second run: SOURCE_LANG='en', TARGET_LANG='sw'
SOURCE_LANG = 'sw'
TARGET_LANG = 'en'

# Common Voice Swahili — flat structure (only validated clips uploaded)
# For En->Sw run, set SOURCE_LANG='en' and CV_DATASET_DIR to FLEURS or leave as-is
# (FLEURS English is downloaded automatically from HuggingFace when source=fleurs)
CV_DATASET_DIR = '/kaggle/input/cv-swahili'  # clips/ and validated.tsv are directly here
CV_SPLIT = 'validated'

# TTS model for the TARGET language — None uses pretrained MMS-TTS for both directions
VITS_MODEL_DIR = None

# Processing limits (set None for all data)
MAX_SAMPLES = 50000

print(f'Direction: {SOURCE_LANG} -> {TARGET_LANG}')
print(f'Dataset: {CV_DATASET_DIR}')
print(f'VITS model: {VITS_MODEL_DIR or "pretrained MMS-TTS"}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Verify the dataset is attached and accessible
if not os.path.exists(CV_DATASET_DIR):
    print(f'ERROR: Dataset not found at {CV_DATASET_DIR}')
    print('Make sure you attached the correct Kaggle Dataset before running.')
    print('Go to: Notebook Settings -> Add Data -> search for cv-swahili (or cv-english)')
else:
    import subprocess
    result = subprocess.run(['wc', '-l', f'{CV_DATASET_DIR}/{CV_SPLIT}.tsv'], capture_output=True, text=True)
    print(result.stdout.strip())
    clips = os.listdir(f'{CV_DATASET_DIR}/clips')
    print(f'Clips directory: {len(clips)} files')
    print(f'Sample clips: {clips[:5]}')
    print('Dataset ready!')

## Step 1: Whisper Transcription (~2-3 hrs for 50K samples)

Transcribe source audio with word-level timestamps.
These timestamps are critical for the contextual alignment step.

In [ ]:
TRANSCRIPTION_DIR = f'{BASE_DIR}/transcriptions/{SOURCE_LANG}'

# Check if we can resume from a previous run
existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSCRIPTION_DIR) else 0
print(f'Existing transcriptions: {existing}')

!python {REPO_DIR}/data/prepare/transcribe_whisper.py \
    --source common_voice \
    --lang {SOURCE_LANG} \
    --split {CV_SPLIT} \
    --dataset_dir {CV_DATASET_DIR} \
    --output_dir {TRANSCRIPTION_DIR} \
    --whisper_model medium \
    --max_samples {MAX_SAMPLES} \
    --resume_from {existing}

In [ ]:
# Verify transcriptions
import json
from pathlib import Path

trans_files = sorted(Path(TRANSCRIPTION_DIR).glob('*.json'))
print(f'Total transcription files: {len(trans_files)}')

# Preview a sample
if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Text: {sample.get("text", "")[:100]}')
    print(f'  Duration: {sample.get("audio_duration", 0):.1f}s')
    print(f'  Words: {len(sample.get("words", []))}')
    if sample.get('words'):
        print(f'  First word: {sample["words"][0]}')

## Step 2: MADLAD Translation (~1-2 hrs)

Translate all transcriptions to the target language using MADLAD-400-3B.

In [ ]:
DIRECTION = f'{SOURCE_LANG}2{TARGET_LANG}'
TRANSLATION_DIR = f'{BASE_DIR}/translations/{DIRECTION}'

existing = len([f for f in os.listdir(TRANSLATION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSLATION_DIR) else 0
print(f'Existing translations: {existing}')

!python {REPO_DIR}/data/prepare/translate_madlad.py \
    --input_dir {TRANSCRIPTION_DIR} \
    --output_dir {TRANSLATION_DIR} \
    --direction {DIRECTION} \
    --batch_size 32 \
    --max_samples {MAX_SAMPLES} \
    --resume_from {existing}

In [ ]:
# Verify translations
trans_files = sorted(Path(TRANSLATION_DIR).glob('*.json'))
print(f'Total translation files: {len(trans_files)}')

if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Source ({SOURCE_LANG}): {sample.get("source_text", "")[:100]}')
    print(f'  Target ({TARGET_LANG}): {sample.get("translated_text", "")[:100]}')

## Step 3: Contextual Alignment (~30 min)

Compute per-word alignment between source and target text using
MADLAD-400 perplexity (Hibiki paper, Section 3.2.1, eq. 6).

This determines which source word maps to which target word,
which is needed for the silence insertion step.

In [ ]:
ALIGNMENT_DIR = f'{BASE_DIR}/alignments/{DIRECTION}'

!python {REPO_DIR}/data/prepare/run_pipeline.py \
    --source common_voice \
    --source_lang {SOURCE_LANG} \
    --target_lang {TARGET_LANG} \
    --dataset_dir {CV_DATASET_DIR} \
    --base_dir {BASE_DIR} \
    --max_samples {MAX_SAMPLES} \
    --step align

In [ ]:
# Verify alignments
align_files = sorted(Path(ALIGNMENT_DIR).glob('*.json'))
print(f'Total alignment files: {len(align_files)}')

if align_files:
    with open(align_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {align_files[0].name}')
    print(f'  Source: {sample.get("source_text", "")[:80]}')
    print(f'  Target: {sample.get("translated_text", "")[:80]}')
    print(f'  Alignment pairs: {len(sample.get("alignment", []))}')
    if sample.get('alignment'):
        print(f'  First 5 pairs: {sample["alignment"][:5]}')

## Step 4: TTS Synthesis + Silence Insertion (~2-3 hrs)

This is the key step that creates the synthetic parallel speech:
1. Synthesize target-language speech from translations using VITS/MMS-TTS
2. Get word timestamps on synthesized audio with Whisper
3. Apply silence insertion for causal alignment

**For Sw->En:** Uses pretrained `facebook/mms-tts-eng` (no fine-tuning needed)

**For En->Sw:** Uses your fine-tuned VITS from `00a_stage0_vits.ipynb`

In [ ]:
SYNTH_DIR = f'{BASE_DIR}/synthetic_audio/{DIRECTION}'

# Build the command
cmd = f"""python {REPO_DIR}/data/prepare/synthesize_tts.py \\
    --translation_dir {TRANSLATION_DIR} \\
    --output_dir {SYNTH_DIR} \\
    --target_lang {TARGET_LANG} \\
    --alignment_dir {ALIGNMENT_DIR} \\
    --whisper_model medium \\
    --target_sr 24000 \\
    --max_samples {MAX_SAMPLES}"""

if VITS_MODEL_DIR:
    cmd += f" \\
    --vits_model_dir {VITS_MODEL_DIR}"

print(f'Running synthesis...\n{cmd}\n')
!{cmd}

In [ ]:
# Verify synthesized audio
import IPython.display as ipd
import torchaudio

aligned_dir = f'{SYNTH_DIR}/aligned_audio'
raw_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

# Check which directory has output
wav_dir = aligned_dir if os.path.exists(aligned_dir) else raw_dir
wav_files = sorted(Path(wav_dir).glob('*.wav'))[:5]

print(f'Synthesized WAVs in {wav_dir}: {len(list(Path(wav_dir).glob("*.wav")))} total')
print()

for wav_path in wav_files:
    waveform, sr = torchaudio.load(str(wav_path))
    dur = waveform.shape[1] / sr
    print(f'{wav_path.name} ({dur:.2f}s, {sr}Hz)')
    ipd.display(ipd.Audio(waveform.squeeze().numpy(), rate=sr))

## Step 5: Encode Audio with Mimi Codec

Encode both source audio (from Common Voice) and synthesized target audio
through the Mimi neural codec to get discrete token representations.

Output: `.npy` files of shape `(Q=8, T)` — 8 codebooks x T timesteps.

In [ ]:
AUDIO_TOKEN_DIR = f'{BASE_DIR}/audio_tokens'

# Encode source audio (Common Voice)
print('=== Encoding source audio (Common Voice) ===')
!python {REPO_DIR}/data/prepare/encode_audio.py \
    --source common_voice \
    --lang {SOURCE_LANG} \
    --split {CV_SPLIT} \
    --dataset_dir {CV_DATASET_DIR} \
    --output_dir {AUDIO_TOKEN_DIR}/cv_{SOURCE_LANG} \
    --num_codebooks 8 \
    --max_samples {MAX_SAMPLES} \
    --max_duration 20.0

In [ ]:
# Encode synthesized target audio
# The aligned audio is already at 24kHz, encode via directory mode
synth_wav_dir = f'{SYNTH_DIR}/aligned_audio'
if not os.path.exists(synth_wav_dir):
    synth_wav_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

print(f'=== Encoding synthesized {TARGET_LANG} audio ===')
!python {REPO_DIR}/data/prepare/encode_audio.py \
    --source directory \
    --audio_dir {synth_wav_dir} \
    --output_dir {AUDIO_TOKEN_DIR}/synth_{TARGET_LANG} \
    --num_codebooks 8 \
    --max_duration 30.0

In [ ]:
# Verify encoded tokens
import numpy as np

for subdir in [f'cv_{SOURCE_LANG}', f'synth_{TARGET_LANG}']:
    token_dir = f'{AUDIO_TOKEN_DIR}/{subdir}'
    if os.path.exists(token_dir):
        npy_files = list(Path(token_dir).glob('*.npy'))
        print(f'\n{subdir}: {len(npy_files)} token files')
        if npy_files:
            sample = np.load(str(npy_files[0]))
            print(f'  Sample shape: {sample.shape} (Q={sample.shape[0]}, T={sample.shape[1]})')
            print(f'  Duration: ~{sample.shape[1] / 12.5:.1f}s at 12.5Hz')

## Step 6: Pipeline Summary

Check all outputs and prepare for training.

In [ ]:
print('=' * 60)
print(f'PIPELINE COMPLETE: {SOURCE_LANG} -> {TARGET_LANG}')
print('=' * 60)

artifacts = {
    'Transcriptions': f'{BASE_DIR}/transcriptions/{SOURCE_LANG}',
    'Translations': f'{BASE_DIR}/translations/{DIRECTION}',
    'Alignments': f'{BASE_DIR}/alignments/{DIRECTION}',
    'Synthesized audio': SYNTH_DIR,
    f'Source tokens (cv_{SOURCE_LANG})': f'{AUDIO_TOKEN_DIR}/cv_{SOURCE_LANG}',
    f'Target tokens (synth_{TARGET_LANG})': f'{AUDIO_TOKEN_DIR}/synth_{TARGET_LANG}',
}

for name, path in artifacts.items():
    if os.path.exists(path):
        n_json = len(list(Path(path).rglob('*.json')))
        n_wav = len(list(Path(path).rglob('*.wav')))
        n_npy = len(list(Path(path).rglob('*.npy')))
        parts = []
        if n_json: parts.append(f'{n_json} json')
        if n_wav: parts.append(f'{n_wav} wav')
        if n_npy: parts.append(f'{n_npy} npy')
        print(f'  {name}: {", ".join(parts)}')
    else:
        print(f'  {name}: [NOT FOUND]')

print(f'\n=== Next Steps ===')
if SOURCE_LANG == 'sw':
    print('1. Re-run this notebook with SOURCE_LANG="en", TARGET_LANG="sw"')
    print('   (set VITS_MODEL_DIR to your trained Swahili VITS model)')
else:
    print('1. Both directions are done!')
print('2. Run notebook 00_data_preparation.ipynb for tokenizer + text tokens')
print('3. Create S2ST manifest with create_s2st_manifest.py')
print('4. Start Stages 1-2 training on Kaggle')